# External paired-dataset evaluation
Evaluates only the exact original 22.91 FSD checkpoint with fixed per-image latents, alpha 0, and LPIPS enabled.

In [ ]:
!pip install -q lpips scikit-image
from pathlib import Path
import hashlib, shutil, subprocess, sys
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/External_Evaluation')

roots = [p.parent for p in INPUT.rglob('run_external_evaluation.py')
         if (p.parent/'src/evaluation.py').is_file()]
if len(roots) != 1:
    raise RuntimeError(f'Expected one external-evaluation source dataset, found {roots}')
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(roots[0], WORK)

EXPECTED = '5d7bac873b0b915fe6c0679b103fea9afe25f70de3958cab8da3d8779d156a37'
checkpoints = []
for path in INPUT.rglob('student_distilled.pth'):
    if hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED:
        checkpoints.append(path)
if len(checkpoints) != 1:
    raise RuntimeError(f'Exact 22.91 checkpoint not found: {checkpoints}')
CHECKPOINT = checkpoints[0]

data_roots = []
for path in INPUT.rglob('*'):
    if path.is_dir() and (path/'Huawei/low').is_dir() and any(path.rglob('VE-LOL-L-Syn-Low_test')):
        data_roots.append(path)
if len(data_roots) != 1:
    raise RuntimeError(f'Expected one uploaded Zip_file dataset root, found {data_roots}')
DATA_ROOT = data_roots[0]
print('Source:', roots[0])
print('Checkpoint:', CHECKPOINT)
print('Data:', DATA_ROOT)


In [ ]:
cmd = [sys.executable, str(WORK/'run_external_evaluation.py'),
       '--data-root', str(DATA_ROOT), '--checkpoint', str(CHECKPOINT),
       '--seed', '99173']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(WORK/'results/external_summary.csv'))
EVIDENCE = Path('/kaggle/working/external_evaluation_evidence')
if EVIDENCE.exists(): shutil.rmtree(EVIDENCE)
EVIDENCE.mkdir()
for path in (WORK/'results').rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.csv', '.json'}:
        target = EVIDENCE/path.relative_to(WORK/'results')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
archive = shutil.make_archive('/kaggle/working/external_evaluation_evidence', 'zip', EVIDENCE)
print('Download:', archive)
